# 01 — Historical Data Cleaning

Loads all 9 raw historical sources, applies the supervisor-mandated imputation strategy (no countries dropped), merges into a single panel, and saves `data/processed/historical_panel.csv` plus a data quality report.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import modules
sys.path.insert(0, str(Path.cwd().parent))   # works when notebook is in src/notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from preprocessing import (
    YEARS_HIST,
    load_wb_historical,
    load_hdi,
    compute_na_pct,
    impute_variable,
    merge_to_panel,
    build_quality_report,
)
from config import DATA_RAW_DIR, DATA_PROCESSED_DIR

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Raw dir:       ", DATA_RAW_DIR)
print("Processed dir: ", DATA_PROCESSED_DIR)

## 1. Load raw sources

World Bank CSVs use `skiprows=3`. HDI has a different structure and is handled separately.

In [ ]:
# ── World Bank CSVs ───────────────────────────────────────────────────────────
gdp_raw   = load_wb_historical(DATA_RAW_DIR / "GDP_HistoricalData_1960_2024.csv",                              "gdp")
corr_raw  = load_wb_historical(DATA_RAW_DIR / "ControlOfCorruption_Historical_WorldBank_1996-2023.csv",         "control_of_corruption")
agri_raw  = load_wb_historical(DATA_RAW_DIR / "EmploymentInAgriculture_Historical_WorldBank_1991-2023.csv",     "employment_agriculture")
gini_raw  = load_wb_historical(DATA_RAW_DIR / "GiniCoefficient_Historical_WorldBank_1963-2024.csv",             "gini_coefficient")
pov3_raw  = load_wb_historical(DATA_RAW_DIR / "POV_3$_1963_2024.csv",                                          "poverty_3")
pov420_raw = load_wb_historical(DATA_RAW_DIR / "POV_4.20$_1963_2024.csv",                                      "poverty_4_20")
pov830_raw = load_wb_historical(DATA_RAW_DIR / "POV_8.30$_1963_2024.csv",                                      "poverty_8_30")
pov10_raw  = load_wb_historical(DATA_RAW_DIR / "POV_10$_1963_2024.csv",                                        "poverty_10")

# ── HDI (different format, with extrapolation 2020–2023) ──────────────────────
hdi_raw = load_hdi(DATA_RAW_DIR / "HDI_1990_2019.csv")

print("Loaded sources:")
for name, df in [("GDP", gdp_raw), ("HDI", hdi_raw), ("Corruption", corr_raw),
                 ("Agri", agri_raw), ("Gini", gini_raw),
                 ("Pov$3", pov3_raw), ("Pov$4.20", pov420_raw),
                 ("Pov$8.30", pov830_raw), ("Pov$10", pov10_raw)]:
    print(f"  {name:<12} {len(df):>6} rows  |  {df['country_name'].nunique()} countries")

## 2. HDI extrapolation check

Verify that the 2019→2023 geometric-mean extrapolation looks reasonable for a sample of countries.

In [ ]:
sample_countries = ["Germany", "Brazil", "Nigeria", "India", "Afghanistan"]
hdi_check = hdi_raw[hdi_raw["country_name"].isin(sample_countries)]

fig, ax = plt.subplots(figsize=(10, 4))
for country, grp in hdi_check.groupby("country_name"):
    grp = grp.sort_values("year")
    ax.plot(grp["year"], grp["hdi"], marker="o", markersize=3, label=country)
    ax.axvline(2019.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.text(2019.6, ax.get_ylim()[0] + 0.01, "extrapolated →", fontsize=7, color="grey")

ax.set_title("HDI — raw (1993–2019) + extrapolated (2020–2023)")
ax.set_xlabel("Year")
ax.set_ylabel("HDI")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Missingness before imputation

Compute original NA fractions per variable. These are saved for the quality report and for flagging high-imputation countries.

In [ ]:
sources = {
    "gdp":                    (gdp_raw,    "gdp"),
    "hdi":                    (hdi_raw,    "hdi"),
    "control_of_corruption":  (corr_raw,   "control_of_corruption"),
    "employment_agriculture": (agri_raw,   "employment_agriculture"),
    "gini_coefficient":       (gini_raw,   "gini_coefficient"),
    "poverty_3":              (pov3_raw,   "poverty_3"),
    "poverty_4_20":           (pov420_raw, "poverty_4_20"),
    "poverty_8_30":           (pov830_raw, "poverty_8_30"),
    "poverty_10":             (pov10_raw,  "poverty_10"),
}

na_pcts_before = {}
rows = []
for varname, (df, col) in sources.items():
    pct = compute_na_pct(df, col)
    na_pcts_before[varname] = pct
    rows.append({
        "variable":    varname,
        "n_countries": len(pct),
        "pct_missing_mean": round(pct.mean() * 100, 1),
        "pct_missing_max":  round(pct.max() * 100, 1),
        "n_complete":       (pct == 0).sum(),
        "n_gt50pct_na":     (pct > 0.5).sum(),
    })

missing_summary = pd.DataFrame(rows)
print(missing_summary.to_string(index=False))

# Heatmap of missingness by variable
fig, ax = plt.subplots(figsize=(11, 3.5))
heat_data = pd.DataFrame({
    v: na_pcts_before[v] for v in na_pcts_before
}).fillna(1.0)
sns.heatmap(heat_data.T, cmap="YlOrRd", vmin=0, vmax=1,
            xticklabels=False, yticklabels=True, ax=ax,
            cbar_kws={"label": "Fraction missing"})
ax.set_title("Missingness heatmap — all variables (each row = variable, each col = country)")
ax.set_xlabel("Countries (sorted alphabetically)")
plt.tight_layout()
plt.show()

## 4. Imputation

**Strategy (per supervisor feedback — no countries dropped):**
1. Linear interpolation per country (interior gaps)
2. Forward-fill → backward-fill (max 3 years, edge NaNs)
3. Regional median (WB region via ISO3 code) for remaining NaNs

Poverty data is intentionally sparse (survey-based every few years) — interpolation is standard practice here.

In [ ]:
gdp_imp   = impute_variable(gdp_raw,    "gdp")
hdi_imp   = impute_variable(hdi_raw,    "hdi")
corr_imp  = impute_variable(corr_raw,   "control_of_corruption")
agri_imp  = impute_variable(agri_raw,   "employment_agriculture")
gini_imp  = impute_variable(gini_raw,   "gini_coefficient")
pov3_imp  = impute_variable(pov3_raw,   "poverty_3")
pov420_imp = impute_variable(pov420_raw,"poverty_4_20")
pov830_imp = impute_variable(pov830_raw,"poverty_8_30")
pov10_imp  = impute_variable(pov10_raw, "poverty_10")

# Verify: NaN counts before vs after
for varname, raw_df, imp_df, col in [
    ("gdp",                   gdp_raw,    gdp_imp,    "gdp"),
    ("hdi",                   hdi_raw,    hdi_imp,    "hdi"),
    ("control_of_corruption", corr_raw,   corr_imp,   "control_of_corruption"),
    ("employment_agriculture",agri_raw,   agri_imp,   "employment_agriculture"),
    ("gini_coefficient",      gini_raw,   gini_imp,   "gini_coefficient"),
    ("poverty_3",             pov3_raw,   pov3_imp,   "poverty_3"),
    ("poverty_4_20",          pov420_raw, pov420_imp, "poverty_4_20"),
    ("poverty_8_30",          pov830_raw, pov830_imp, "poverty_8_30"),
    ("poverty_10",            pov10_raw,  pov10_imp,  "poverty_10"),
]:
    n_before = raw_df[col].isna().sum()
    n_after  = imp_df[col].isna().sum()
    print(f"  {varname:<28}  NaN before: {n_before:>5}  →  after: {n_after:>5}")

## 5. Spot-check imputation

Visualise original vs. imputed values for a poverty-sparse country (e.g. Nigeria) and a data-rich country (Germany).

In [ ]:
def plot_imputation_check(raw_df, imp_df, value_col, country, title_suffix=""):
    r = raw_df[raw_df["country_name"] == country].sort_values("year")
    i = imp_df[imp_df["country_name"] == country].sort_values("year")
    
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(i["year"], i[value_col], label="Imputed", linewidth=2, color="#2196F3")
    ax.scatter(r["year"], r[value_col], label="Original (observed)", color="#E91E63",
               zorder=5, s=20)
    ax.set_title(f"{country} — {value_col}{title_suffix}")
    ax.set_xlabel("Year")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

plot_imputation_check(pov3_raw, pov3_imp, "poverty_3", "Nigeria",
                      " (sparse survey data — interpolation expected)")
plot_imputation_check(pov3_raw, pov3_imp, "poverty_3", "Brazil")
plot_imputation_check(gini_raw, gini_imp, "gini_coefficient", "India")

## 6. Merge into panel

Outer-join all imputed sources on `country_name` × `year`. `high_imputation_flag` is set `True` for any country that had >50% original NaNs in at least one variable.

In [ ]:
panel_raw = merge_to_panel(
    gdp_raw, hdi_raw, corr_raw, agri_raw, gini_raw,
    pov3_raw, pov420_raw, pov830_raw, pov10_raw,
    na_pcts=na_pcts_before,
)

panel = merge_to_panel(
    gdp_imp, hdi_imp, corr_imp, agri_imp, gini_imp,
    pov3_imp, pov420_imp, pov830_imp, pov10_imp,
    na_pcts=na_pcts_before,
)

print(f"Panel shape:          {panel.shape}")
print(f"Countries:            {panel['country_name'].nunique()}")
print(f"Year range:           {panel['year'].min()}–{panel['year'].max()}")
print(f"High-imputation flag: {panel.drop_duplicates('country_name')['high_imputation_flag'].sum()} countries")
panel.head()

## 7. Retention: new approach vs. old '>10 NAs → drop' approach

Document how many countries we keep compared to the naive drop strategy.

In [ ]:
# Simulate the old approach: drop countries with >10 NaNs in ANY source over 1993–2023
# (31 years = 10/31 ≈ 32% threshold)
old_threshold_n = 10
n_years = len(YEARS_HIST)

def count_retained_old_approach(df, value_col):
    na_counts = (
        df[df["year"].isin(YEARS_HIST)]
        .groupby("country_name")[value_col]
        .apply(lambda s: s.isna().sum())
    )
    return (na_counts <= old_threshold_n).sum()

print("Countries retained per variable:")
print(f"{'Variable':<28}  Old (drop >10 NA)  New (impute all)")
for varname, (df, col) in sources.items():
    old = count_retained_old_approach(df, col)
    new = df["country_name"].nunique()
    print(f"  {varname:<26}  {old:>17}  {new:>16}")

print(f"
Total unique countries in new panel: {panel['country_name'].nunique()}")
print("The 'old approach' would disproportionately drop Sub-Saharan African")
print("and South Asian countries — exactly those with highest poverty rates.")

## 8. High-imputation-flag countries

List countries with >50% original NaNs in at least one variable. These are retained in the dataset but flagged for the report.

In [ ]:
flagged = (
    panel.drop_duplicates("country_name")
    .query("high_imputation_flag == True")[["country_name", "country_code"]]
    .sort_values("country_name")
    .reset_index(drop=True)
)
print(f"Flagged countries ({len(flagged)} total):")
print(flagged.to_string())

## 9. Data quality report

In [ ]:
quality_report = build_quality_report(panel_raw, panel, na_pcts_before)

# Bar chart: % missing before vs after imputation
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(quality_report))
w = 0.35
ax.bar([i - w/2 for i in x], quality_report["pct_missing_before"], width=w,
       label="Before imputation", color="#EF5350")
ax.bar([i + w/2 for i in x], quality_report["pct_missing_after"],  width=w,
       label="After imputation",  color="#42A5F5")
ax.set_xticks(list(x))
ax.set_xticklabels(quality_report["variable"], rotation=30, ha="right", fontsize=9)
ax.set_ylabel("% missing")
ax.set_title("Missing data before vs. after imputation")
ax.legend()
ax.grid(True, axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 10. Save outputs

In [ ]:
# Main panel
out_panel = DATA_PROCESSED_DIR / "historical_panel.csv"
panel.to_csv(out_panel, index=False)
print(f"Saved: {out_panel}  ({len(panel):,} rows)")

# Quality report
out_report = DATA_PROCESSED_DIR / "data_quality_report.csv"
quality_report.to_csv(out_report, index=False)
print(f"Saved: {out_report}")

# Flagged countries list
out_flags = DATA_PROCESSED_DIR / "high_imputation_countries.csv"
flagged.to_csv(out_flags, index=False)
print(f"Saved: {out_flags}  ({len(flagged)} countries flagged)")

print("\nFinal panel column dtypes:")
print(panel.dtypes)